<a href="https://colab.research.google.com/github/Arif0000/Pytorch/blob/main/optuna_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 7.1 MB/s eta 0:00:00


In [2]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd



In [3]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
import numpy as np

cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df [cols_with_missing_vals].replace(0,np.nan)

df.fillna(df.mean(), inplace=True)
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [15]:
from pandas.core.tools.datetimes import Scalar
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train.shape)
print(X_test.shape)

(537, 8)
(231, 8)


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trail):
  n_estimator = trail.suggest_int('n_estimators', 50, 200)
  max_depth = trail.suggest_int('max_depth',3,20)

  model = RandomForestClassifier(n_estimators= n_estimator, max_depth= max_depth, random_state=42)

  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

  return score

In [12]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())

study.optimize(objective, n_trials=50)

[I 2026-03-04 10:23:11,347] A new study created in memory with name: no-name-ef6e8e1e-0add-4e05-a177-5328d144eaba
[I 2026-03-04 10:23:12,448] Trial 0 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 184, 'max_depth': 7}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-03-04 10:23:13,578] Trial 1 finished with value: 0.7802607076350093 and parameters: {'n_estimators': 118, 'max_depth': 16}. Best is trial 1 with value: 0.7802607076350093.
[I 2026-03-04 10:23:15,038] Trial 2 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 150, 'max_depth': 11}. Best is trial 1 with value: 0.7802607076350093.
[I 2026-03-04 10:23:16,426] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 196, 'max_depth': 10}. Best is trial 1 with value: 0.7802607076350093.
[I 2026-03-04 10:23:17,219] Trial 4 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 140, 'max_depth': 4}. Best is trial 1 with value: 0.780260

In [13]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7858472998137803
Best hyperparameters: {'n_estimators': 121, 'max_depth': 15}


In [16]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.76


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [18]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-03-04 10:26:02,560] A new study created in memory with name: no-name-6e7fff64-f2ae-4e6d-b6c3-b6dbbfce0957
[I 2026-03-04 10:26:03,830] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 87, 'max_depth': 9}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-03-04 10:26:05,103] Trial 1 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 113, 'max_depth': 11}. Best is trial 1 with value: 0.7653631284916201.
[I 2026-03-04 10:26:08,022] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 157, 'max_depth': 18}. Best is trial 2 with value: 0.7709497206703911.
[I 2026-03-04 10:26:08,691] Trial 3 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 52, 'max_depth': 5}. Best is trial 3 with value: 0.7746741154562384.
[I 2026-03-04 10:26:09,648] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 93, 'max_depth': 5}. Best is trial 3 with value: 0.7746741154

In [19]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 118, 'max_depth': 7}


In [20]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.75


In [21]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}


In [22]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-03-04 10:27:33,236] A new study created in memory with name: no-name-d9b53102-18de-47c6-b765-0eddf40da985
[I 2026-03-04 10:27:34,081] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-03-04 10:27:36,357] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-03-04 10:27:37,635] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-03-04 10:27:39,790] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-03-04 10:27:42,059] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [23]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [24]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


In [25]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [26]:
# 1. Optimization History
plot_optimization_history(study).show()

In [27]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()


In [28]:
# 3. Slice Plot
plot_slice(study).show()

In [29]:
# 4. Contour Plot
plot_contour(study).show()

In [30]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [31]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [32]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [33]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-03-04 10:30:47,094] A new study created in memory with name: no-name-47bb60cf-8147-42c7-8a1b-624d0541b6da
[I 2026-03-04 10:30:47,150] Trial 0 finished with value: 0.7765363128491621 and parameters: {'classifier': 'SVM', 'C': 0.14813622362243214, 'kernel': 'sigmoid', 'gamma': 'scale'}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-03-04 10:30:47,195] Trial 1 finished with value: 0.74487895716946 and parameters: {'classifier': 'SVM', 'C': 0.5593086566370349, 'kernel': 'sigmoid', 'gamma': 'scale'}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-03-04 10:30:47,231] Trial 2 finished with value: 0.6945996275605214 and parameters: {'classifier': 'SVM', 'C': 9.062073128939797, 'kernel': 'sigmoid', 'gamma': 'auto'}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-03-04 10:30:51,705] Trial 3 finished with value: 0.7541899441340782 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 226, 'learning_rate': 0.06700305841863906, 'max_depth': 9, 'min_s

In [34]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)
study.trials_dataframe()

Best trial parameters: {'classifier': 'SVM', 'C': 0.13491422535954664, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [35]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,80
GradientBoosting,10
RandomForest,10


In [36]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.748045
RandomForest,0.764060
SVM,0.776816


In [37]:
# 1. Optimization History
plot_optimization_history(study).show()

In [38]:
# 3. Slice Plot
plot_slice(study).show()

In [39]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [42]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2026-03-04 10:34:23,208] A new study created in memory with name: no-name-d5b34ff5-9b4a-49f7-876b-34d5851b66cc


[0]	train-mlogloss:1.06130	eval-mlogloss:1.06428
[1]	train-mlogloss:1.01700	eval-mlogloss:1.01518
[2]	train-mlogloss:0.96786	eval-mlogloss:0.96259
[3]	train-mlogloss:0.92148	eval-mlogloss:0.91514
[4]	train-mlogloss:0.88039	eval-mlogloss:0.87182
[5]	train-mlogloss:0.83912	eval-mlogloss:0.82848
[6]	train-mlogloss:0.80293	eval-mlogloss:0.79047
[7]	train-mlogloss:0.77182	eval-mlogloss:0.76046
[8]	train-mlogloss:0.73947	eval-mlogloss:0.72555
[9]	train-mlogloss:0.70791	eval-mlogloss:0.69225
[10]	train-mlogloss:0.67910	eval-mlogloss:0.66363
[11]	train-mlogloss:0.65196	eval-mlogloss:0.63470
[12]	train-mlogloss:0.63552	eval-mlogloss:0.61840
[13]	train-mlogloss:0.61065	eval-mlogloss:0.59108
[14]	train-mlogloss:0.59557	eval-mlogloss:0.57461
[15]	train-mlogloss:0.57443	eval-mlogloss:0.55257
[16]	train-mlogloss:0.55542	eval-mlogloss:0.53300
[17]	train-mlogloss:0.53946	eval-mlogloss:0.51728
[18]	train-mlogloss:0.52218	eval-mlogloss:0.49962
[19]	train-mlogloss:0.50641	eval-mlogloss:0.48406
[20]	train

[I 2026-03-04 10:34:25,142] Trial 0 finished with value: 1.0 and parameters: {'lambda': 0.0012164703947697255, 'alpha': 0.13314893991574867, 'eta': 0.04235012961350021, 'gamma': 0.0017143784740760264, 'max_depth': 3, 'min_child_weight': 3, 'subsample': 0.4152043367772039, 'colsample_bytree': 0.5327628311945874}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:0.93190	eval-mlogloss:0.93770
[1]	train-mlogloss:0.74310	eval-mlogloss:0.73306
[2]	train-mlogloss:0.60437	eval-mlogloss:0.58413
[3]	train-mlogloss:0.49661	eval-mlogloss:0.47255
[4]	train-mlogloss:0.41505	eval-mlogloss:0.38603
[5]	train-mlogloss:0.35060	eval-mlogloss:0.31403
[6]	train-mlogloss:0.30152	eval-mlogloss:0.26301
[7]	train-mlogloss:0.26262	eval-mlogloss:0.22262
[8]	train-mlogloss:0.23414	eval-mlogloss:0.18981
[9]	train-mlogloss:0.21220	eval-mlogloss:0.16452
[10]	train-mlogloss:0.19652	eval-mlogloss:0.14755
[11]	train-mlogloss:0.18979	eval-mlogloss:0.13764
[12]	train-mlogloss:0.18035	eval-mlogloss:0.12754
[13]	train-mlogloss:0.17680	eval-mlogloss:0.12254
[14]	train-mlogloss:0.17583	eval-mlogloss:0.12310
[15]	train-mlogloss:0.17237	eval-mlogloss:0.11768
[16]	train-mlogloss:0.17137	eval-mlogloss:0.11497
[17]	train-mlogloss:0.16708	eval-mlogloss:0.10937
[18]	train-mlogloss:0.16466	eval-mlogloss:0.10618
[19]	train-mlogloss:0.16398	eval-mlogloss:0.10671
[20]	train

[I 2026-03-04 10:34:27,883] Trial 1 finished with value: 1.0 and parameters: {'lambda': 0.053404790073419806, 'alpha': 4.565446228428123e-07, 'eta': 0.2006477021425775, 'gamma': 3.369866396516876e-08, 'max_depth': 4, 'min_child_weight': 6, 'subsample': 0.7833412371547624, 'colsample_bytree': 0.685383302408495}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:1.04907	eval-mlogloss:1.04748
[1]	train-mlogloss:1.01612	eval-mlogloss:1.01014


[I 2026-03-04 10:34:59,584] Trial 48 pruned. Trial was pruned at iteration 2.


[0]	train-mlogloss:0.87778	eval-mlogloss:0.87761
[1]	train-mlogloss:0.67495	eval-mlogloss:0.66839


[I 2026-03-04 10:34:59,914] Trial 49 pruned. Trial was pruned at iteration 2.


Best trial: {'lambda': 0.0012164703947697255, 'alpha': 0.13314893991574867, 'eta': 0.04235012961350021, 'gamma': 0.0017143784740760264, 'max_depth': 3, 'min_child_weight': 3, 'subsample': 0.4152043367772039, 'colsample_bytree': 0.5327628311945874}
Best accuracy: 1.0


In [41]:
! pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 3.4 MB/s eta 0:00:00


In [43]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()